# B9W9 Interim Submission: Time Series Forecasting for Portfolio Management Optimization

This notebook covers Task 1 and initial Task 2 progress for the 10 Academy Week 9 interim submission. It extracts TSLA, BND, and SPY data from yfinance, cleans and explores the data, runs stationarity and risk analysis, then builds an initial ARIMA forecasting baseline for Tesla.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(ROOT))

import pandas as pd
import matplotlib.pyplot as plt

from src.data_loader import DEFAULT_TICKERS, START_DATE, END_DATE, load_local_or_fetch
from src.preprocessing import clean_price_data, add_return_features, combine_adjusted_close
from src.risk import value_at_risk, sharpe_ratio, detect_return_outliers
from src.stationarity import adf_test
from src.modeling import chronological_split, fit_arima_forecast, forecast_metrics

FIGURES = ROOT / 'figures'
REPORTS = ROOT / 'reports'
PROCESSED = ROOT / 'data' / 'processed'
for p in [FIGURES, REPORTS, PROCESSED]:
    p.mkdir(parents=True, exist_ok=True)

print(DEFAULT_TICKERS, START_DATE, END_DATE)

## 1. Data Extraction

The required assets are TSLA, BND, and SPY. The code uses `yfinance` from 2015-01-01 to 2026-07-01 because yfinance treats the end date as exclusive, which covers market data through 2026-06-30 when available.

In [ ]:
raw_data = load_local_or_fetch(PROCESSED)
{ticker: df.shape for ticker, df in raw_data.items()}

## 2. Data Cleaning and Understanding

Cleaning steps include sorting by date, removing duplicated dates, converting OHLCV columns to numeric format, and handling missing values using time interpolation plus forward/backward fill.

In [ ]:
clean_data = {}
for ticker, df in raw_data.items():
    cleaned = clean_price_data(df)
    clean_data[ticker] = add_return_features(cleaned)
    print(f'\n{ticker} dtypes:')
    print(clean_data[ticker].dtypes)
    print('Missing values after cleaning:', int(clean_data[ticker].isna().sum().sum()))

summary = pd.concat({ticker: df.describe() for ticker, df in clean_data.items()})
summary

## 3. EDA Visualization 1: Adjusted Closing Price Over Time

This chart compares long-term price behavior across TSLA, BND, and SPY. TSLA is expected to show the strongest growth and highest instability, while BND is expected to behave more defensively.

In [ ]:
prices = combine_adjusted_close(clean_data)
prices.to_csv(PROCESSED / 'adj_close_prices.csv')
ax = prices.plot(figsize=(12, 6), title='Adjusted Closing Price Over Time')
ax.set_xlabel('Date')
ax.set_ylabel('Adjusted Close Price (USD)')
plt.tight_layout()
plt.savefig(FIGURES / 'closing_price_over_time.png', dpi=150)
plt.show()

## 4. EDA Visualization 2: Daily Percentage Change

Daily returns make volatility easier to compare across assets. Large spikes indicate abnormal trading days and potential market shocks.

In [ ]:
returns = prices.pct_change().dropna()
returns.to_csv(PROCESSED / 'daily_returns.csv')
ax = returns.plot(figsize=(12, 6), title='Daily Percentage Change')
ax.set_xlabel('Date')
ax.set_ylabel('Daily Return')
plt.tight_layout()
plt.savefig(FIGURES / 'daily_percentage_change.png', dpi=150)
plt.show()

## 5. EDA Visualization 3: Rolling Mean and Rolling Volatility

The 30-day rolling mean smooths short-term noise, while rolling standard deviation of returns captures changing volatility over time.

In [ ]:
tsla = clean_data['TSLA']
fig, ax1 = plt.subplots(figsize=(12, 6))
ax1.plot(tsla.index, tsla['Adj Close'], label='TSLA Adj Close')
ax1.plot(tsla.index, tsla['Rolling_Mean_30'], label='30-Day Rolling Mean')
ax1.set_title('TSLA Adjusted Close and 30-Day Rolling Mean')
ax1.set_xlabel('Date')
ax1.set_ylabel('Price')
ax1.legend()
plt.tight_layout()
plt.savefig(FIGURES / 'tsla_rolling_mean.png', dpi=150)
plt.show()

tsla['Rolling_Std_30'].plot(figsize=(12, 4), title='TSLA 30-Day Rolling Volatility')
plt.xlabel('Date')
plt.ylabel('Rolling Std of Daily Returns')
plt.tight_layout()
plt.savefig(FIGURES / 'tsla_rolling_volatility.png', dpi=150)
plt.show()

## 6. Outlier Detection

Unusually high or low return days are identified using absolute z-scores of daily returns greater than or equal to 3. These points represent days that may require market-event interpretation.

In [ ]:
outlier_tables = {}
for ticker in DEFAULT_TICKERS:
    outliers = detect_return_outliers(clean_data[ticker]['Daily_Return'], z_threshold=3.0)
    outlier_tables[ticker] = outliers
    outliers.to_csv(PROCESSED / f'{ticker}_return_outliers.csv')
    print(f'{ticker}: {len(outliers)} outlier days')

outlier_tables['TSLA'].head(10)

## 7. Stationarity Testing with Augmented Dickey-Fuller

The ADF null hypothesis is that the series has a unit root, meaning it is non-stationary. A p-value below 0.05 suggests stationarity. For financial assets, adjusted close prices are usually non-stationary, while daily returns are usually closer to stationary. This is why ARIMA uses differencing (`d=1`).

In [ ]:
adf_rows = []
for ticker in DEFAULT_TICKERS:
    adf_rows.append(adf_test(clean_data[ticker]['Adj Close'], f'{ticker} adjusted close'))
    adf_rows.append(adf_test(clean_data[ticker]['Daily_Return'], f'{ticker} daily return'))

adf_results = pd.DataFrame(adf_rows)
adf_results.to_csv(REPORTS / 'stationarity_results.csv', index=False)
adf_results[['series', 'adf_statistic', 'p_value', 'stationary_at_5pct', 'interpretation']]

## 8. Risk Metrics: VaR and Sharpe Ratio

The 95% historical one-day VaR estimates the loss threshold exceeded in the worst 5% of days. The Sharpe Ratio measures annualized risk-adjusted return using daily returns.

In [ ]:
risk_rows = []
for ticker in DEFAULT_TICKERS:
    risk_rows.append({
        'ticker': ticker,
        'VaR_95_daily': value_at_risk(clean_data[ticker]['Daily_Return'], confidence_level=0.95),
        'Sharpe_Ratio_annualized': sharpe_ratio(clean_data[ticker]['Daily_Return'])
    })

risk_metrics = pd.DataFrame(risk_rows)
risk_metrics.to_csv(REPORTS / 'risk_metrics.csv', index=False)
risk_metrics

## 9. Task 2 Initial Progress: Chronological Train/Test Split

The TSLA series is split by date instead of random shuffling. Training data is before 2025-01-01 and the test period is 2025 onward, preserving temporal order.

In [ ]:
train, test = chronological_split(clean_data['TSLA']['Adj Close'], split_date='2025-01-01')
print('Train:', train.index.min(), 'to', train.index.max(), train.shape)
print('Test:', test.index.min(), 'to', test.index.max(), test.shape)

## 10. Initial ARIMA Forecast

The interim baseline uses `ARIMA(1,1,1)`. The `d=1` component is justified by the stationarity tests: raw prices usually need differencing, while returns are more stable. More parameter tuning can be added in the final submission using ACF/PACF or `auto_arima`.

In [ ]:
fitted_model, forecast = fit_arima_forecast(train, steps=len(test), order=(1, 1, 1))
forecast.index = test.index
metrics = forecast_metrics(test, forecast)
pd.DataFrame(metrics, index=['ARIMA(1,1,1)']).to_csv(REPORTS / 'model_metrics.csv')
pd.DataFrame(metrics, index=['ARIMA(1,1,1)'])

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(train.index, train.values, label='Train')
plt.plot(test.index, test.values, label='Test')
plt.plot(forecast.index, forecast.values, label='ARIMA Forecast')
plt.title('TSLA ARIMA Forecast vs Test Period')
plt.xlabel('Date')
plt.ylabel('Adjusted Close Price (USD)')
plt.legend()
plt.tight_layout()
plt.savefig(FIGURES / 'tsla_arima_forecast.png', dpi=150)
plt.show()

## Interim Conclusion

Task 1 is complete with data extraction, cleaning, EDA visualizations, outlier detection, ADF testing, VaR, and Sharpe Ratio. Task 2 has initial progress through a chronological split, an ARIMA baseline with documented parameters, test-period forecasts, and evaluation metrics. The final submission can extend this by tuning ARIMA/SARIMA, adding LSTM, optimizing the portfolio, and backtesting against a 60% SPY / 40% BND benchmark.